# Structural audit statistics on a toy table

This notebook plants leakage in a synthetic table and shows how three of the structural and declared-scope statistics from the paper respond. It runs in a few seconds and needs no download.

The functions below are a **short illustrative re-implementation**. The code that produced the paper's features is `nn_features` and its neighbours in `research/stage7_2026-11-02/code/s3_run.py`; use that for anything quantitative.

| Statistic | Definition |
|---|---|
| `exact_eval` | share of evaluation rows that have an exactly equal fit row |
| `excess_t` | share of evaluation rows within radius `t * m` of a fit row, minus the same share among fit rows (leave-one-out); `m` is the median leave-one-out nearest-neighbor distance among fit rows |
| `scope_eval_frac` | share of evaluation rows inside the declared preprocessing scope |

In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors


def structural_stats(X_fit, X_eval, scope_ids=None, eval_ids=None, radii=(0.02, 0.1, 0.3)):
    mu, sd = X_fit.mean(0), X_fit.std(0)
    sd[sd == 0] = 1.0
    Zf, Ze = (X_fit - mu) / sd, (X_eval - mu) / sd                       # fit-standardized space
    d_eval = NearestNeighbors(n_neighbors=1).fit(Zf).kneighbors(Ze)[0][:, 0]
    d_loo = NearestNeighbors(n_neighbors=2).fit(Zf).kneighbors(Zf)[0][:, 1]
    fit_rows = {tuple(r) for r in X_fit}
    m = max(float(np.median(d_loo[d_loo > 0])), 1e-9)
    out = {"exact_eval": float(np.mean([tuple(r) in fit_rows for r in X_eval]))}   # tuple equality, not a distance threshold
    for t in radii:
        out[f"excess_{t}"] = float(np.mean(d_eval / m < t) - np.mean(d_loo / m < t))
    if scope_ids is not None:
        out["scope_eval_frac"] = len(set(scope_ids) & set(eval_ids)) / len(eval_ids)
    return out

## A clean, deduplicated table

Rows are continuous, so no two are equal: this is the *deduplicated* setting, where an equal cross-split row can only be a planted copy.

In [2]:
rng = np.random.default_rng(104729)
n, p = 2000, 8
X = rng.normal(size=(n, p))
y = (X[:, 0] + 0.5 * X[:, 1] + rng.normal(size=n) > 0).astype(int)
ids = np.arange(n)
eval_ids, fit_ids = ids[:500], ids[500:]
X_fit, X_eval = X[fit_ids], X[eval_ids]

rows = {"clean": structural_stats(X_fit, X_eval, scope_ids=fit_ids, eval_ids=eval_ids)}
pd.DataFrame(rows).T.round(3)

,exact_eval,excess_0.02,excess_0.1,excess_0.3,scope_eval_frac
clean,0.0,0.0,0.0,0.0,0.0


## Plant copies, near copies and a leaky scope

- **Exact copies:** 16 evaluation rows are appended to the fit rows.
- **Near copies:** the same, with noise of 0.05 column standard deviations.
- **Leaky scope:** the scaler is declared as fitted on fit rows plus half of the evaluation rows. The data are unchanged.
- **Look-alike:** 16 fit rows are duplicated inside the fit set. Nothing crosses the split, so this is *not* leakage.

In [3]:
k = 16
copies = X_eval[:k]
near = copies + rng.normal(scale=0.05 * X.std(0), size=copies.shape)
self_dup = X_fit[:k]

rows["exact copies (k=16)"] = structural_stats(np.vstack([X_fit, copies]), X_eval, fit_ids, eval_ids)
rows["near copies (k=16, sigma=0.05)"] = structural_stats(np.vstack([X_fit, near]), X_eval, fit_ids, eval_ids)
rows["leaky scope (half of evaluation)"] = structural_stats(X_fit, X_eval, np.concatenate([fit_ids, eval_ids[:250]]), eval_ids)
rows["look-alike: self-duplicates"] = structural_stats(np.vstack([X_fit, self_dup]), X_eval, fit_ids, eval_ids)
table = pd.DataFrame(rows).T
table.round(4)

,exact_eval,excess_0.02,excess_0.1,excess_0.3,scope_eval_frac
clean,0.000,0.0000,0.0000,0.0000,0.0
exact copies (k=16),0.032,0.0320,0.0320,0.0320,0.0
"near copies (k=16, sigma=0.05)",0.000,0.0000,0.0140,0.0320,0.0
leaky scope (half of evaluation),0.000,0.0000,0.0000,0.0000,0.5
look-alike: self-duplicates,0.000,-0.0211,-0.0211,-0.0211,0.0


Each planted mechanism moves the statistic built for it and leaves the others near their clean values:

- exact copies raise `exact_eval` to 16/500;
- near copies leave `exact_eval` at zero and raise the excess at radii 0.1 and 0.3;
- the leaky scope moves only `scope_eval_frac`, because the data are identical to the clean table. Without a declared scope, no data-only statistic can see it;
- the self-duplicate look-alike moves none of them upward.

## Why raw tables are harder

Now let the table contain natural duplicates: distinct source records with identical predictors. `exact_eval` is already positive with no leakage, so a rule that fires on any cross-split equality fires on the clean table. Under an *ancestry* definition of leakage those rows are not leaks, and no method that sees only values can tell an individual equal pair from a copy.

In [4]:
X_raw = np.round(X, 0)[:, :3]                 # coarse values: many natural duplicates
Xr_fit, Xr_eval = X_raw[fit_ids], X_raw[eval_ids]
raw = {"raw, clean": structural_stats(Xr_fit, Xr_eval),
       "raw, 16 exact copies": structural_stats(np.vstack([Xr_fit, Xr_eval[:k]]), Xr_eval)}
pd.DataFrame(raw).T[["exact_eval"]].round(3)

,exact_eval
"raw, clean",0.960
"raw, 16 exact copies",0.962


The two rows are almost the same: `exact_eval` is already 0.96 before anything is planted. This is the identifiability limit of Section 6.1 of the paper, and the reason every table in the suite is evaluated twice, raw and deduplicated.

## The refit-based audit

The behavioral audits in the paper do not read these statistics. They edit the pipeline, refit, and compare the change in loss with a placebo edit of the same size. That code is in `src/leakagelens/interventions/`:

- `SuspectedPipeline` holds what an auditor may see and refuses any field that identifies an injection;
- `apply_rule` applies a corrective edit (`I1` removes training rows equal to a held-out row, `I4` refits preprocessing on training rows only) or a placebo (`P1`, `P5`);
- `decide_episode` fires when a correction moves the loss more than its placebo.

In [5]:
from leakagelens.interventions import RuleConfig, SuspectedPipeline, apply_rule

features = pd.DataFrame(np.vstack([X_fit, copies, X_eval]), columns=[f"f{i}" for i in range(p)])
target = pd.Series(np.concatenate([y[fit_ids], y[eval_ids][:k], y[eval_ids]]))
train = np.arange(len(X_fit) + k)
test = np.arange(len(X_fit) + k, len(features))
pipeline = SuspectedPipeline(features=features, target=target, train_idx=train, validation_idx=np.array([], dtype=int),
                             test_idx=test, preprocessor_fit_idx=train, model_family="logistic_regression",
                             model_backend="sklearn", model_params={}, model_seed=0)
config = RuleConfig(proxy_screen_threshold=0.9, proxy_screen_bins=10, placebo_row_fraction=0.05, placebo_seed_offset=1000)
edited, change = apply_rule(pipeline, "I1", config)
print("training rows removed by I1:", change.rows_removed_from_training)

training rows removed by I1: 16


The edit finds exactly the 16 planted rows. Whether *refitting* without them moves the loss more than removing 16 random rows is the behavioral question, and the paper finds that response weak: 16 copies are a small share of the evaluation rows, so their effect on the loss is diluted.